# Experiment 9: Implementation of K-Means Clustering


### Step 1: Import Libraries & Generate Customer Segmentation Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score

### Step 2: Elbow Method & Silhouette Analysis to Find Optimal K

In [ ]:
# Realistic Customer Segmentation (Annual Income vs Spending Score -> 4 natural segments)
X_cust, _ = make_blobs(n_samples=250, centers=[[-1.5,-1.5], [1.5,1.5], [-1.5,1.5], [1.5,-1.5]],
                       cluster_std=0.45, random_state=42)
X_cust[:, 0] = np.round(X_cust[:, 0] * 20 + 60, 1)  # Annual Income (k$)
X_cust[:, 1] = np.round(np.clip(X_cust[:, 1] * 20 + 50, 1, 99), 1)  # Spending Score (1-100)

scaler = StandardScaler()
X_sc = scaler.fit_transform(X_cust)

wcss, sil_scores = [], []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = km.fit_predict(X_sc)
    wcss.append(km.inertia_)
    sil_scores.append(silhouette_score(X_sc, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.5))
ax1.plot(K_range, wcss, 'o-', color='#4F46E5', lw=2.5)
ax1.axvline(4, color='#E11D48', linestyle='--', label='Elbow at K=4')
ax1.set_title('Elbow Method (WCSS / Inertia vs K)', fontweight='bold')
ax1.set_xlabel('Number of Clusters (K)'); ax1.set_ylabel('WCSS'); ax1.legend(); ax1.grid(True, alpha=0.4)

ax2.plot(K_range, sil_scores, 's-', color='#0D9488', lw=2.5)
ax2.axvline(4, color='#E11D48', linestyle='--', label='Peak Silhouette at K=4')
ax2.set_title('Silhouette Score vs K', fontweight='bold')
ax2.set_xlabel('Number of Clusters (K)'); ax2.set_ylabel('Silhouette Score'); ax2.legend(); ax2.grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

### Step 3: Fit Final K-Means (K=4) & Visualize Clusters + Centroids

In [ ]:
kmeans = KMeans(n_clusters=4, init='k-means++', n_init=10, random_state=42)
clusters = kmeans.fit_predict(X_sc)
centroids_orig = scaler.inverse_transform(kmeans.cluster_centers_)

plt.figure(figsize=(8.5, 5.5))
colors = ['#4F46E5', '#0D9488', '#D97706', '#E11D48']
for c in range(4):
    mask = (clusters == c)
    plt.scatter(X_cust[mask, 0], X_cust[mask, 1], s=60, color=colors[c], alpha=0.75, label=f'Segment {c+1}')

plt.scatter(centroids_orig[:, 0], centroids_orig[:, 1], s=260, marker='*', color='gold',
            edgecolor='black', linewidths=1.5, zorder=5, label='Centroids')
plt.title('K-Means Customer Segmentation (K = 4)', fontsize=13, fontweight='bold')
plt.xlabel('Annual Income (k$)'); plt.ylabel('Spending Score (1-100)')
plt.legend(); plt.grid(True, linestyle='--', alpha=0.4); plt.show()

print(f"Final WCSS (Inertia): {kmeans.inertia_:.2f}")
print(f"Silhouette Score    : {silhouette_score(X_sc, clusters):.4f}")

### Step 4: Compare K-Means vs. Agglomerative Clustering on Iris Dataset

In [ ]:
iris = load_iris()
X_iris_sc = StandardScaler().fit_transform(iris.data)

km_iris = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(X_iris_sc)
hc_iris = AgglomerativeClustering(n_clusters=3, linkage='ward').fit_predict(X_iris_sc)

print(f"K-Means Silhouette       : {silhouette_score(X_iris_sc, km_iris):.4f} | ARI vs True Labels: {adjusted_rand_score(iris.target, km_iris):.4f}")
print(f"Agglomerative Silhouette : {silhouette_score(X_iris_sc, hc_iris):.4f} | ARI vs True Labels: {adjusted_rand_score(iris.target, hc_iris):.4f}")

### Conclusion
- **Elbow Method (WCSS)** and **Silhouette Score** consistently pinpoint optimal cluster count $K = 4$.
- Scaling features prior to K-Means ensures Euclidean distance treats all attributes equitably.